In [ ]:
cus_raw = session.table(f"{DB}.{RAW}.STREAM_T_CUSTODIES").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
ind_cols = ["TWENTY_NINE_C_IND"]
code_cols = ["LEGAL_STATUS_CODE", "END_REASON_CODE", "CUSTODY_TYPE"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["SOURCE_FILE_NAME"]

all_cols = [c.name for c in cus_raw.schema.fields]
select_exprs = []
for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in ind_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c))
    elif c in code_cols:
        select_exprs.append(upper(trim(col(c))).alias(c))
    elif c in user_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c))
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

cus_clean = cus_raw.select(select_exprs)
print("Transformations applied")

In [ ]:
valid_cus = cus_clean.filter(col("CUS_ID").is_not_null())
invalid_cus = cus_clean.filter(col("CUS_ID").is_null()).with_column("ERROR_REASON", lit("CUS_ID_NULL"))
print("Valid/invalid split defined")

In [ ]:
checksum_cols = [
    ("CUS_ID", "number"), ("LEGAL_STATUS_CODE", "string"), ("TWENTY_NINE_C_IND", "string"),
    ("CUSTODY_TYPE", "string"), ("END_REASON_CODE", "string"),
    ("ADD_USER", "string"), ("MOD_USER", "string"),
    ("PERSON_PERSON_ID", "number"), ("ORGN_ORGN_ID", "number"),
    ("VPA_VPA_ID", "number"), ("SUR_SUR_ID", "number"),
    ("CAP_CAP_ID", "number"), ("POI_POI_ID", "number"),
    ("PERSON_PERSON_CHILD_ID", "number"), ("TICKLER_ID", "number"),
    ("MOD_CAP_CAP_ID", "number"), ("START_DATE", "timestamp"),
    ("END_DATE", "timestamp"), ("CHILD_ADJUDICATED_DATE", "timestamp"),
    ("TWENTY_NINE_C_DATE", "timestamp"), ("REASONABLE_EFFORT_DATE", "timestamp")
]
checksum_exprs = []
for col_name, col_type in checksum_cols:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

cus_final = valid_cus.with_column(
    "CHECKSUM", sha2(concat_ws(lit("|"), *checksum_exprs), 256)
).with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())

cus_final.write.save_as_table(f"{DB}.{STG}.{STG_CUSTODIES}", mode="truncate")
print(f"CUSTODIES loaded into {STG}.{STG_CUSTODIES}")

In [ ]:
invalid_count = invalid_cus.count()
if invalid_count > 0:
    invalid_cus.select(
        lit("T_CUSTODIES").alias("TABLE_NAME"), col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"), col("RAW_LOAD_TIMESTAMP")
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_CUSTODIES}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_CUSTODIES_LOAD', 'STAGING',
    datetime.now(), datetime.now(), rows_processed, invalid_count, status, None)
session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_CUSTODIES_LOAD', 'STAGING',
    f'CUSTODIES staging completed. Rows: {rows_processed}, Failed: {invalid_count}')
print(f"STG_CUSTODIES complete | Valid: {rows_processed} | Invalid: {invalid_count}")